# 8.10 — Sequence-to-sequence encoder--decoder

Sequence-to-sequence learning turns one variable-length stream into another by encoding what was read and decoding what should be said next.

This lesson assembles sequence modeling into translation-shaped work: read a source sequence, then generate a target sequence. Autoregressive probability factors the target into one next-token decision at a time.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np

SEED = 87
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def softmax(scores, axis=-1):
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=axis, keepdims=True)


def pad_sequences(sequences, pad_value=0):
    width = max(len(seq) for seq in sequences)
    out = np.full((len(sequences), width), pad_value, dtype=int)
    mask = np.zeros((len(sequences), width), dtype=float)
    for row, seq in enumerate(sequences):
        out[row, : len(seq)] = seq
        mask[row, : len(seq)] = 1.0
    return out, mask


def summarize_rung(rung):
    lengths = [len(seq) for seq in rung["sequences"]]
    vocab = sorted({token for seq in rung["sequences"] for token in seq})
    return {
        "name": rung["name"],
        "n": len(rung["sequences"]),
        "min_len": min(lengths),
        "max_len": max(lengths),
        "vocab": vocab,
    }


def build_sequence_classification_ladder(kind):
    base = [
        {"name": "D1 lesson toy", "sequences": [[1, 0, 1, 1]], "labels": [1], "dependency": 4},
        {"name": "D2 clean patterns", "sequences": [[1, 0, 1], [0, 0, 1], [1, 1, 0], [0, 1, 0], [1, 0, 0]], "labels": [1, 0, 1, 0, 0], "dependency": 3},
        {"name": "D3 long gaps and noise", "sequences": [[1, 0, 0, 1], [1, 2, 0, 0], [0, 2, 2, 1], [0, 0, 0, 0], [1, 0, 2, 1], [0, 1, 2, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 4},
        {"name": "D4 click/dialogue snippets", "sequences": [[3, 1, 0, 4], [2, 0, 4], [3, 2, 2, 4], [1, 1, 0, 0], [3, 0, 1, 4], [2, 2, 0, 4]], "labels": [1, 0, 1, 0, 1, 0], "dependency": 5},
        {"name": "D5 delayed dependency", "sequences": [[1, 0, 0, 0, 0, 4], [0, 0, 1, 0, 0, 4], [1, 2, 2, 2, 2, 0], [0, 2, 2, 2, 2, 4], [1, 0, 2, 0, 2, 4], [0, 0, 0, 0, 0, 0]], "labels": [1, 0, 0, 0, 1, 0], "dependency": 6},
    ]
    if kind == "lstm":
        base[0] = {"name": "D1 lesson gate toy", "sequences": [[2, 0, 2]], "labels": [1], "dependency": 3}
    if kind == "gru":
        base[0] = {"name": "D1 lesson interpolation toy", "sequences": [[2, 1, 0, 1]], "labels": [1], "dependency": 4}
    return base


def build_seq2seq_ladder():
    return [
        {"name": "D1 lesson reverse", "pairs": [([1, 2, 3], [3, 2, 1])], "length": 3},
        {"name": "D2 clean copy/reverse", "pairs": [([1, 2], [2, 1]), ([2, 3], [3, 2]), ([1, 1, 2], [2, 1, 1]), ([3, 1], [1, 3]), ([2, 2, 3], [3, 2, 2])], "length": 3},
        {"name": "D3 unequal reorder", "pairs": [([4, 1, 2], [2, 1]), ([4, 2, 3, 1], [1, 3, 2]), ([1, 4], [1]), ([2, 1, 3], [3, 1, 2]), ([3, 4, 2], [2, 3])], "length": 4},
        {"name": "D4 command pairs", "pairs": [([5, 1, 9], [9, 1]), ([5, 2, 8], [8, 2]), ([6, 1, 7], [7, 1]), ([5, 3, 9], [9, 3]), ([6, 2, 8], [8, 2])], "length": 3},
        {"name": "D5 longer delayed EOS", "pairs": [([1, 0, 0, 2, 9], [2, 1, 10]), ([3, 0, 4, 0, 9], [4, 3, 10]), ([2, 2, 0, 1, 9], [1, 2, 2, 10]), ([5, 0, 0, 0, 6, 9], [6, 5, 10]), ([7, 1, 0, 8, 9], [8, 1, 7, 10])], "length": 6},
    ]


def build_attention_ladder():
    return [
        {"name": "D1 illustrative QKV", "source": [1, 2, 3], "targets": [1], "gold": [1]},
        {"name": "D2 clean alignments", "source": [1, 2, 3, 4], "targets": [1, 2, 3, 4, 2], "gold": [0, 1, 2, 3, 1]},
        {"name": "D3 distractor reorder", "source": [5, 1, 9, 2, 3], "targets": [1, 2, 3], "gold": [1, 3, 4]},
        {"name": "D4 QA translation toy", "source": [7, 4, 8, 4, 9, 2], "targets": [4, 9, 2], "gold": [1, 4, 5]},
        {"name": "D5 diffuse distractors", "source": [8, 0, 4, 0, 9, 0, 2, 0, 4, 1], "targets": [9, 2, 1, 4], "gold": [4, 6, 9, 2]},
    ]


def build_transformer_ladder():
    return [
        {"name": "D1 3-token toy", "sequences": [[1, 2, 1]], "labels": [1], "length": 3},
        {"name": "D2 clean sentence patterns", "sequences": [[1, 3, 2], [2, 3, 1], [1, 4, 4], [2, 4, 4], [1, 3, 3]], "labels": [1, 0, 1, 0, 1], "length": 3},
        {"name": "D3 order conflicts", "sequences": [[5, 1, 9, 2], [5, 2, 9, 1], [1, 0, 2, 0], [2, 0, 1, 0], [1, 9, 9, 2]], "labels": [1, 0, 1, 0, 1], "length": 4},
        {"name": "D4 inline classification corpus", "sequences": [[7, 1, 4, 2, 8], [7, 2, 4, 1, 8], [1, 6, 6, 2, 9], [2, 6, 6, 1, 9], [1, 4, 4, 2, 8]], "labels": [1, 0, 1, 0, 1], "length": 5},
        {"name": "D5 longer sequences", "sequences": [[1, 0, 0, 3, 0, 2], [2, 0, 0, 3, 0, 1], [9, 1, 0, 0, 2, 8], [9, 2, 0, 0, 1, 8], [1, 4, 0, 4, 0, 2]], "labels": [1, 0, 1, 0, 1], "length": 6},
    ]


def accuracy(predictions, labels):
    predictions = np.asarray(predictions)
    labels = np.asarray(labels)
    return float(np.mean(predictions == labels))


## The concept, built once (D1)

Seq2seq models use an encoder summary and an autoregressive decoder: $$P(y_{1:T}\mid x_{1:S})=\prod_{t=1}^{T}P(y_t\mid y_{1:t-1},\operatorname{Enc}(x_{1:S}))$$.

In [ ]:

def encode_scalar(src):
    h = 0.0
    states = []
    for x_t in src:
        h = math.tanh(0.6 * h + 0.4 * x_t)
        states.append(h)
    return h, np.asarray(states)

final_state, encoder_states = encode_scalar([2, 1, 3])
expected = np.asarray([0.664036770267849, 0.6631536913995937, 0.9213506585709649])
assert np.allclose(encoder_states, expected)
assert np.isclose(final_state, 0.9213506585709649)
encoder_states


The decoder unfolds target positions one by one. Teacher forcing scores next-token predictions at each shifted target position, while free running can compound errors.

In [ ]:

def decode_scalar(h0, steps):
    h = h0
    states = []
    for step in range(steps):
        h = math.tanh(0.7 * h)
        states.append(h)
    return np.asarray(states)

decoder_states = decode_scalar(0.9, 4)
expected_decoder = np.asarray([0.5580522155596244, 0.3719088632753982, 0.254609968964449, 0.1763635334341262])
assert np.allclose(decoder_states, expected_decoder)
assert np.isclose(0.9 ** 10, 0.3486784401000001)
print(decoder_states)


## The dataset ladder

The inline F7 ladder starts with a reverse toy and grows toward unequal length, command-like pairs, and delayed EOS.

In [ ]:

rungs = build_seq2seq_ladder()
for rung in rungs:
    lengths = [(len(src), len(tgt)) for src, tgt in rung["pairs"]]
    print(rung["name"], "pairs", len(rung["pairs"]), "lengths", lengths[:2], "sample", rung["pairs"][0])


## Run the same method across D1--D5

The basic decoder uses the final encoder vector as a bottleneck. The fixed version adds an attention-style lookup: choose source tokens directly when reversing.

In [ ]:

def seq2seq_table_translate(src, use_attention_lookup=False, max_steps=None):
    final_state, states = encode_scalar(src)
    clean = [token for token in src if token not in [0, 9]]
    if use_attention_lookup:
        output = list(reversed(clean))
    else:
        output = list(reversed(clean[-3:]))
    if 9 in src:
        output.append(10)
    if max_steps is not None:
        output = output[:max_steps]
    alignment = np.zeros((len(output), len(src)))
    for row, token in enumerate(output):
        matches = [idx for idx, src_token in enumerate(src) if src_token == token]
        if matches:
            alignment[row, matches[-1]] = 1.0
    return output, final_state, alignment

results = []
for rung in rungs:
    correct = []
    for src, tgt in rung["pairs"]:
        pred, state, alignment = seq2seq_table_translate(src, use_attention_lookup=False, max_steps=len(tgt))
        correct.append(int(pred == tgt))
    results.append({"rung": rung["name"], "length": rung["length"], "accuracy": float(np.mean(correct))})

for row in results:
    print(f'{row["rung"]:28s} length={row["length"]:2d} exact_accuracy={row["accuracy"]:.3f}')


## Results visualization

The top row shows source-target alignment panels. The metric curve shows exact-sequence accuracy against output length.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(15, 5))
for index, rung in enumerate(rungs):
    src, tgt = rung["pairs"][0]
    pred, state, alignment = seq2seq_table_translate(src, use_attention_lookup=True, max_steps=len(tgt))
    axes[0, index].imshow(alignment, aspect="auto", cmap="Blues")
    axes[0, index].set_title(rung["name"].split()[0])
    axes[0, index].set_xlabel("source")
    axes[0, index].set_ylabel("target")

axes[1, 0].plot([row["length"] for row in results], [row["accuracy"] for row in results], marker="o")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].set_xlabel("output/source length")
axes[1, 0].set_ylabel("exact accuracy")
axes[1, 0].set_title("metric curve")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on D5: bottleneck and exposure bias

A single final state can lose early tokens, and free-running decisions compound: the lesson's $0.9^{10}=0.3486784401000001$ makes the compounding visible.

In [ ]:

d5 = rungs[-1]
bottleneck = []
attention_fixed = []
for src, tgt in d5["pairs"]:
    pred_bad, _, _ = seq2seq_table_translate(src, use_attention_lookup=False, max_steps=len(tgt))
    pred_good, _, _ = seq2seq_table_translate(src, use_attention_lookup=True, max_steps=len(tgt))
    bottleneck.append(int(pred_bad == tgt))
    attention_fixed.append(int(pred_good == tgt))
print("wrong bottleneck D5 exact accuracy", float(np.mean(bottleneck)))
print("fixed lookup D5 exact accuracy", float(np.mean(attention_fixed)))
print("ten-step no-error probability", 0.9 ** 10)


## Evaluate it + Practice

- Metric: exact-sequence accuracy.
- No-skill baseline: copy or reverse only the last few tokens.
- Cheap sanity check: source [2, 1, 3] encodes to 0.9213506585709649.
- Ablation: disable lookup attention on D5.
- Failure signals: fixed-length decoding truncates or overruns EOS.

Practice 1: Change one rung so the dependency is one step farther away and predict how the metric curve should move.

Practice 2: Insert EOS earlier and verify the decoder stops at it.

Practice 3: Shift teacher-forcing targets by one wrong position and inspect correctness.